# Muhtemel Aşk - haftalık altyazı

1. Üst menü: **Çalışma zamanı > Çalışma zamanı türünü değiştir > L4 GPU** (yoksa T4).
2. Aşağıdaki hücreyi çalıştır (▶). Drive izni sorarsa onayla.
3. "BITTI" yazınca Claude'a **"15. bölüm hazır"** de. Endonezce altyazıyı Claude yazıp videonun yanına koyar.
4. **Bağlantı koparsa, sekme kapanırsa, hata olursa:** aynı hücreyi tekrar çalıştır. Kaldığı yerden devam eder (video tekrar inmez, Whisper ara kayıttan devam eder). Tüm log: `_is/Exx/log.txt`.

- `BOLUM = 0` bir sonraki bölümü kendisi bulur. Show TV'de bulamazsa `URL` alanına bölüm sayfasının linkini yapıştır.
- `WHISPER_ZORLA` işaretliyse Show TV'nin Türkçe altyazısı hiç kullanılmaz, her zaman Whisper çalışır.
- `TEST` işaretliyse her şey gerçek akışla aynı, ama video `TEST` klasörüne, iş dosyaları `_is/TEST_Exx`'e gider. `Sezon 1`'e dokunmaz, otomatik çeviri tetiklenmez.
- `DAKIKA` 0 ise bölümün tamamı, 15 ise sadece ilk 15 dakika (hızlı deneme).
- `VOKAL_AYIR`: Whisper'dan önce müziği ayıklar (Demucs). 24 Eylül testinde sonucu kötüleştirdi, kapalı bırak.

In [ ]:
BOLUM = 0  #@param {type:"integer"}
URL = ""  #@param {type:"string"}
WHISPER_ZORLA = True  #@param {type:"boolean"}
TEST = False  #@param {type:"boolean"}
DAKIKA = 0  #@param {type:"integer"}
VOKAL_AYIR = False  #@param {type:"boolean"}

import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'UYARI: GPU yok! Calisma zamani turunu L4/T4 yap.')
# 23 Eylul'de bu Colab hesabinda GPU ile calistigi dogrulanan paket surumleri
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faster-whisper==1.2.1', 'ctranslate2==4.8.1', 'onnxruntime==1.30.0', 'av==18.1.0',
                'huggingface-hub==1.32.0', 'tokenizers==0.23.2', 'numpy==2.2.6',
                'nvidia-cublas-cu12==12.8.4.1', 'nvidia-cudnn-cu12==9.10.2.21'], check=True)
if VOKAL_AYIR:  # Demucs; kurulamazsa sorun degil, normal sesle devam edilir
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'demucs==4.0.1', 'dora-search==0.1.13', 'openunmix==1.3.0'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'julius', 'einops', 'omegaconf', 'retrying', 'submitit', 'treetable'])

from google.colab import drive, runtime
drive.mount('/content/drive')
sys.path.insert(0, '/content/drive/MyDrive/Muhtemel Ask/sistem')
import importlib, ma_sub
importlib.reload(ma_sub)

try:
    sonuc = ma_sub.run(BOLUM, URL, force_asr=WHISPER_ZORLA, test=TEST, minutes=DAKIKA,
                      vocals=VOKAL_AYIR)
finally:
    drive.flush_and_unmount()   # hata olsa bile log ve ara kayitlar Drive'a yazilsin
print('Drive senkron tamam. GPU kapatiliyor (kota harcanmasin).')
runtime.unassign()
